# This is the primary project file. 


### This file runs through several models to get initial baseline scores. Then processes wine data, and gets scores based on several data processing ways, to determine best scores in attempting to predict if a wine will be reviewed as "good or bad".


License: UC Irvine Machine Learning Repository

This dataset is licensed under a Creative Commons Attribution 4.0 International (CC BY 4.0) license.
This allows for the sharing and adaptation of the datasets for any purpose, provided that the appropriate credit is given.

DOI: 10.24432/C56S3T

## Install required dependencies and read in csv data

In [1]:
# Import dependencies
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
import models as tests
from sklearn.utils import resample
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# Read in html data 
red_df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv',sep=';')
white_df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv',sep=';')

red_df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
...,...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5


In [3]:
# Set column for red and white
red_df['wine_type'] = 'red'
white_df['wine_type'] = 'white'

In [4]:
# combine red and white dataframes
df = pd.concat([red_df, white_df], axis=0)
df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6,white
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5,white
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6,white
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7,white


## Create dataframes for baseline scores and processed scores

In [5]:
# Read in CSV data to run initial models for baseline results
df_initial = df.copy()

# Check column names, null values and Dtypes
df_initial.info()


<class 'pandas.core.frame.DataFrame'>
Index: 6497 entries, 0 to 4897
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         6497 non-null   float64
 1   volatile acidity      6497 non-null   float64
 2   citric acid           6497 non-null   float64
 3   residual sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free sulfur dioxide   6497 non-null   float64
 6   total sulfur dioxide  6497 non-null   float64
 7   density               6497 non-null   float64
 8   pH                    6497 non-null   float64
 9   sulphates             6497 non-null   float64
 10  alcohol               6497 non-null   float64
 11  quality               6497 non-null   int64  
 12  wine_type             6497 non-null   object 
dtypes: float64(11), int64(1), object(1)
memory usage: 710.6+ KB


In [6]:
# Read in CSV data to attempt sampling for better results
df_sample = df.copy()

# Check column names, null values and Dtypes
df_sample.info()


<class 'pandas.core.frame.DataFrame'>
Index: 6497 entries, 0 to 4897
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         6497 non-null   float64
 1   volatile acidity      6497 non-null   float64
 2   citric acid           6497 non-null   float64
 3   residual sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free sulfur dioxide   6497 non-null   float64
 6   total sulfur dioxide  6497 non-null   float64
 7   density               6497 non-null   float64
 8   pH                    6497 non-null   float64
 9   sulphates             6497 non-null   float64
 10  alcohol               6497 non-null   float64
 11  quality               6497 non-null   int64  
 12  wine_type             6497 non-null   object 
dtypes: float64(11), int64(1), object(1)
memory usage: 710.6+ KB


## Run initial data through models to get initial baseline scores

In [7]:
# set X and y variables to determine initial baseline score
X0 = df_initial.drop(columns=['quality','wine_type'])
y0 = df_initial['quality']

In [8]:
# Split data into training and testing data
X_train0,X_test0,y_train0,y_test0 = train_test_split(X0,y0,random_state=13)

In [9]:
# Run initial data through many_models to get baseline scores
# Run data through 'many_models_full' function to determine scores and initial parameter optimization for Random Forest Classifier Model
models_full_preprocess = tests.many_models_full(X_train0,y_train0,X_test0,y_test0)


Random Forest 
Test Accuracy: 0.6726153846153846
balanced test score: 0.3510074145712444
classification report: 
               precision    recall  f1-score   support

           3       0.00      0.00      0.00         7
           4       0.64      0.15      0.24        48
           5       0.70      0.74      0.72       528
           6       0.64      0.77      0.70       705
           7       0.71      0.46      0.56       282
           8       0.95      0.33      0.49        54
           9       0.00      0.00      0.00         1

    accuracy                           0.67      1625
   macro avg       0.52      0.35      0.39      1625
weighted avg       0.68      0.67      0.66      1625



c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Gradient Boost 
Test Accuracy: 0.5889230769230769
balanced test score: 0.26994130770726515

 classification report: 
               precision    recall  f1-score   support

           3       0.00      0.00      0.00         7
           4       0.12      0.04      0.06        48
           5       0.63      0.67      0.65       528
           6       0.57      0.71      0.64       705
           7       0.59      0.31      0.41       282
           8       0.89      0.15      0.25        54
           9       0.00      0.00      0.00         1

    accuracy                           0.59      1625
   macro avg       0.40      0.27      0.29      1625
weighted avg       0.59      0.59      0.57      1625


Logistic Regression 
Test Accuracy: 0.540923076923077
balanced test score: 0.21556138896564428
classification report: 
               precision    recall  f1-score   support

           3       0.00      0.00      0.00         7
           4       0.00      0.00      0.00        48


c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being 


Poly Support Vector 
Test Accuracy: 0.5347692307692308
Balanced test score: 0.2061550151975684
classification report: 
               precision    recall  f1-score   support

           3       0.00      0.00      0.00         7
           4       0.00      0.00      0.00        48
           5       0.62      0.54      0.58       528
           6       0.50      0.78      0.61       705
           7       0.52      0.12      0.20       282
           8       0.00      0.00      0.00        54
           9       0.00      0.00      0.00         1

    accuracy                           0.53      1625
   macro avg       0.24      0.21      0.20      1625
weighted avg       0.51      0.53      0.49      1625


ADA low estimators 
Test Accuracy: 0.344
balanced test score: 0.23277674706246135
classification report: 
               precision    recall  f1-score   support

           3       0.01      0.71      0.03         7
           4       0.00      0.00      0.00        48
           

...

c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



SVC Sigmoid 
Test Accuracy: 0.4338461538461538
balanced test score: 0.20026135212305426
classification report: 
               precision    recall  f1-score   support

           3       0.00      0.00      0.00         7
           4       0.08      0.17      0.11        48
           5       0.47      0.46      0.46       528
           6       0.46      0.56      0.50       705
           7       0.43      0.22      0.29       282
           8       0.00      0.00      0.00        54
           9       0.00      0.00      0.00         1

    accuracy                           0.43      1625
   macro avg       0.21      0.20      0.20      1625
weighted avg       0.43      0.43      0.42      1625



c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


non-tuned models dataframe
                     Trained Score  Test Score  Balanced Test Score  \
Model Name                                                            
Random Forest             1.000000    0.672615             0.351007   
Gradient Boosting         0.713054    0.588923             0.269941   
ADA boost                 0.234606    0.227077             0.249291   
ADA Low Estimators        0.366174    0.344000             0.232777   
Logistic Regression       0.542693    0.540923             0.215561   
SVC poly                  0.580665    0.534769             0.206155   
SVC sigmoid               0.433703    0.433846             0.200261   

                     Balanced Difference  
Model Name                                
Random Forest                   0.648993  
Gradient Boosting               0.430798  
ADA boost                      -0.048019  
ADA Low Estimators             -0.046457  
Logistic Regression             0.015383  
SVC poly                        

c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\nick\.conda\envs\dev\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


## Process Data for using in machine learning models

### Bin into good and bad categories

In [ ]:
# Check unique values of quality column
df['quality'].unique()

In [ ]:
# Check how many of each value to pick where to create good/bad thresholds
df['quality'].value_counts(normalize=True)

In [ ]:
# Create threshold for bad category
threshold = 6
df['quality'] = df['quality'].where(df['quality'] > threshold, other=0)

# Check quality column for unique value changes
print(df['quality'].unique())

In [ ]:
# Create threshold for good category
threshold = 7
df['quality'] = df['quality'].where(df['quality'] < threshold, other=1)

# Check quality column for unique value changes
print(df['quality'].unique())

In [ ]:
# Check value percentages
df['quality'].value_counts(normalize=True)

### Sample data to determine if sampling or binning is better

In [ ]:
# Check quality column value percentages to determine sampling
df_sample['quality'].value_counts(normalize=True)

In [ ]:
# Check quality column values
df_sample['quality'].value_counts()

In [ ]:
# Check quality column values mean
df_sample['quality'].value_counts().mean()

In [ ]:
# Set columns as variables for sampling
df_3 = df_sample[df_sample.quality==3]               
df_4 = df_sample[df_sample.quality==4]               
df_5 = df_sample[df_sample.quality==5]     
df_6 = df_sample[df_sample.quality==6]     
df_7 = df_sample[df_sample.quality==7]     
df_8 = df_sample[df_sample.quality==8]
df_9 = df_sample[df_sample.quality==9]     

In [ ]:
# Resample data to equalize values based on mean after removing outliers
df_3_ups = resample(df_3, replace=True, n_samples=1300) 
df_4_ups = resample(df_4, replace=True, n_samples=1300) 
df_7_ups = resample(df_7, replace=True, n_samples=1300) 
df_8_ups = resample(df_8, replace=True, n_samples=1300)
df_9_ups = resample(df_9, replace=True, n_samples=1300)

# Decreases the rows of Majority one's to make balance data
df_5_downs = df_5[df_5.quality==5].sample(n=1300).reset_index(drop=True)
df_6_downs = df_6[df_6.quality==6].sample(n=1300).reset_index(drop=True)


In [ ]:
# Combine sampled columns into dataframe
sampled_q = pd.concat([df_3_ups, df_4_ups, df_7_ups, 
                        df_8_ups, df_9_ups, df_5_downs, df_6_downs]).reset_index(drop=True)

# Display new sampled dataframe
sampled_q

## Check column importance to determine features to use

### Create initial Random Forest Classifier model to determine feature importance to full dataset

In [ ]:
# Create initial Random Forest Classifier Model to determine feature importance
rf_importance = RandomForestClassifier(random_state=13)

# Fit data to initial Random Forest Classifier Model
rf_importance.fit(X_train0, y_train0)

In [ ]:
# Make predictions to determine feature importance
y_train_pred = rf_importance.predict(X_train0)
y_test_pred = rf_importance.predict(X_test0)

In [ ]:
# Create variables to store accuracy_scores and Balanced Accuracy Score
train_accuracy = accuracy_score(y_train0, y_train_pred)
test_accuracy = accuracy_score(y_test0, y_test_pred)
balanced_test = balanced_accuracy_score(y_test0, y_test_pred)

# Show results of initial Random Forest Classifier Model
print(f'\nRandom Forest \nTrain Accuracy: {train_accuracy}\nTest Accuracy: {test_accuracy}\nBalanced Test Score: {balanced_test}')

### Create feature_importance_ instance to View most important features

In [ ]:
# Create variable to store most important features
importances = rf_importance.feature_importances_

# Sort most important features for better usage
importances_sorted = sorted(zip(importances, X0.columns), reverse=True)

# View most important features
print("Most Important Features:")

# Loop through to display features and importance percentage
for importance, feature in importances_sorted[:11]:
    print(f"{feature}: {importance:.4f}")

In [ ]:
# Create variable to plot feature importance
feature_importance_df = pd.DataFrame(importances_sorted, columns=['Importance', 'Feature']).sort_values(by='Importance', ascending=False)

# Plot feature importance
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importance')
plt.show()

### Drop features based on importance

In [ ]:
# Drop columns for sampled set based on feature importance (less than 9% was chosen by testing), and set as X and y variables for sample data
X = sampled_q.drop(columns=['quality','fixed acidity','citric acid','wine_type'])
y = sampled_q['quality']
X, y

In [ ]:
# Drop columns for binned set based on feature importance and correlation, and set as X and y variables for bins models tests
X1 = df.drop(columns=['quality','fixed acidity','citric acid','wine_type'])
y1 = df['quality']
X1, y1

## Use processed data to determine best scores
### split data into training and test variables

In [ ]:
# Split dropped column data into training and testing data for sampled data tests
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=13)

In [ ]:
# Split dropped column data into training and testing data for bins models tests
X_train1,X_test1,y_train1,y_test1 = train_test_split(X1,y1,random_state=13)

### Run bin data through models

In [ ]:
# Run data through many_models_full function to determine scores and optimize Random Forest Classifier Model
print('Binned Data results full')
models_full_bin = tests.many_models_full(X_train1,y_train1,X_test1,y_test1)

### Run sampled data through models

In [ ]:
# Run sampled data through many_models_full function to determine scores and optimize Random Forest Classifier Model
print('Sampled Data results Full')
models_full_sample = tests.many_models_full(X_train,y_train,X_test,y_test)

## Test sampled data combined with binned data through models to determine good/bad wines


### Combine sampled data into bins

In [ ]:
# Create a new dataframe for the combined processing data
combined_df = sampled_q.copy()
# View dataframe
combined_df

In [ ]:
combined_df['quality'].value_counts()

In [ ]:
# Create threshold for bad category
threshold = 6
combined_df['quality'] = combined_df['quality'].where(combined_df['quality'] > threshold, other=0)

# Check quality column for unique value changes
print(combined_df['quality'].unique())

In [ ]:
# Create threshold for good category
threshold = 7
combined_df['quality'] = combined_df['quality'].where(combined_df['quality'] < threshold, other=1)

# Check quality column for unique value changes
print(combined_df['quality'].unique())
print(combined_df['quality'].value_counts())

In [ ]:
# View updated combined Dataframe
combined_df

### split combined data into training and testing data

In [ ]:
# Set X and y variables
X2 = combined_df.drop(['quality','fixed acidity','citric acid','wine_type'],axis=1)
y2 = combined_df['quality']

In [ ]:
# Split into training and testing data
X_train2,X_test2,y_train2,y_test2 = train_test_split(X2,y2,random_state=13)

### Run combined sampled and binned data through models

In [ ]:
# Run combined data through many_models_full function to determine if all parameter settings are worth extra time.
# score and optimize Random Forest Classifier Model
print('Combined Data results full leaf')
models_full_combined = tests.many_models_full(X_train2,y_train2,X_test2,y_test2)

In [ ]:
# Run combined data through many_models_no_leaf function to determine if 'no min/max leaf' parameter setting is worth extra time.
# score and optimize Random Forest Classifier Model
print('Combined Data results no min/max leaf')
models_no_leaf_sample = tests.many_models_no_leaf(X_train2,y_train2,X_test2,y_test2)

In [ ]:
# Run combined data through many_models_min_leaf function to determine if ' min leaf' parameter setting is worth extra time.
# score and optimize Random Forest Classifier Model
print('Combined Data results min leaf')
models_min_leaf_sample = tests.many_models_min_leaf(X_train2,y_train2,X_test2,y_test2)

In [ ]:
# Run combined data through many_models_max_leaf function to determine if 'no max leaf' parameter setting is worth extra time.
# score and optimize Random Forest Classifier Model
print('Combined Data results max leaf')
models_max_leaf_sample = tests.many_models_max_leaf(X_train2,y_train2,X_test2,y_test2)